<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_5_MDanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Trajectory analysis with MDAnalysis

First save a copy of this notebook in your Google Drive (`File -> Save a copy in Drive`).

This is a group report. Each group analyses an MD simulation of a protein with the tools learned in the training notebook (`t05_MDanalysis`) and then uses an experimental structure of the same protein in a complex to ask a specific biological question.

The report has two parts:

1. **General analysis** (sections 3.1–3.4): apply the analyses of the training notebook to your system. Copy and adapt the code from the training notebook; the empty code cells are for you.
2. **System-specific analysis** (section 3.5): the code to set up the analysis is provided; you have to run it, complete it where indicated and interpret the results.

**Group:** 1 / 2

**Members:**

In [ ]:
from datetime import date

today = date.today()
print(today)

## The assignment

### GROUP 1: the angiotensin II type 1 receptor (AT1R)

AT1R is a class A G protein-coupled receptor (GPCR) and the main mediator of the effects of the peptide hormone angiotensin II on blood pressure. It is the target of the angiotensin receptor blockers (ARBs, the "sartans"), widely used against hypertension. Crystal structures are available for the inactive receptor bound to the antagonist ZD7155 ([4YAY](https://www.rcsb.org/structure/4YAY)) and to the inverse agonist olmesartan ([4ZUD](https://www.rcsb.org/structure/4ZUD)), and for the active receptor bound to angiotensin II and stabilised by a nanobody ([6OS0](https://www.rcsb.org/structure/6OS0)).

The receptor was simulated in its inactive state, embedded in a lipid membrane and with a ligand in the binding pocket. The membrane, the ligand and the water were removed from the saved trajectory, which contains the receptor only.

**Question:** how flexible is the ligand-binding pocket, and does the receptor stay in its inactive conformation?

### GROUP 2: Nsp16 from SARS-CoV-2

Nsp16 is the 2'-O-methyltransferase of SARS-CoV-2: it transfers a methyl group from S-adenosylmethionine (SAM) to the cap of the viral mRNA, which helps the virus avoid the innate immune response. Nsp16 is active only in complex with its partner Nsp10. Crystal structures are available for the Nsp16-Nsp10 complex with SAM ([6W4H](https://www.rcsb.org/structure/6W4H)) and with SAM and a cap analogue ([6WVN](https://www.rcsb.org/structure/6WVN)).

Nsp16 was simulated as a monomer, without Nsp10, in water. The water was removed from the saved trajectory, which contains the protein only.

**Question:** how do the SAM-binding site and the surface that binds Nsp10 behave when Nsp16 is alone? What could this tell us about the role of Nsp10?

You can also look at the trajectories with VMD on your laptop: open the PDB file and then, from the menu `File`, choose *Load Data Into Molecule* and select the XTC file.

# Part 1: setup

Run these cells at the start of every session.

In [ ]:
%pip install -q MDAnalysis py3Dmol biopython gdown

We work in a folder of the virtual machine (set `use_drive = True` to work in your Google Drive instead, for example in a folder shared by the group). The cell can be run more than once.

In [ ]:
import os
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()

use_drive = False        # True: keep your files on Google Drive (Colab will ask your permission)
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    workdir = '/content/drive/MyDrive/Structural_Bioinformatics/T05'
else:
    workdir = os.path.join(BASE_DIR, 'T05')
os.makedirs(workdir, exist_ok=True)
%cd $workdir

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # hide some harmless warnings of MDAnalysis

import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
import py3Dmol

Choose your group and download the files of your simulation (only the first time):

In [ ]:
GROUP = 1      # 1 (AT1R) or 2 (Nsp16)

import os
import gdown

files = {1: {'at1r.pdb': '1INr1tNEU71ixIFMXNX9Qq3JrbP45SJss',
             'at1r.xtc': '1MWx4hoboEdWeKusyOwrMtb15nrbIgUep'},
         2: {'nsp16.pdb': '1gGoWWPBrQMpwIO5UxIFgmX4uI9L0tqVZ',
             'nsp16.xtc': '1moxqXDR7pxSsC5rNGPKQS3YKPQt4EJz1'}}

for name, file_id in files[GROUP].items():
    if not os.path.exists(name):
        gdown.download(f'https://drive.google.com/uc?id={file_id}', name, quiet=False)

pdb_file, xtc_file = list(files[GROUP])
print(pdb_file, xtc_file)

We load the trajectory in memory (so that it can be superimposed) and the PDB file alone as reference structure:

In [ ]:
u = mda.Universe(pdb_file, xtc_file, in_memory=True)
ref = mda.Universe(pdb_file)

protein = u.select_atoms('protein')
print(f'{u.atoms.n_atoms} atoms, {protein.n_residues} protein residues '
      f'(numbered from {protein.residues.resids[0]} to {protein.residues.resids[-1]})')
print(f'{u.trajectory.n_frames} frames, one every {u.trajectory.dt:.0f} ps, {u.trajectory.totaltime / 1000:.1f} ns in total')

# Part 2: the report

Write the report in the text cells and add plots with the code cells. Remember that the times in MDAnalysis are in ps and the distances in Å.

## 1. Introduction

*Describe the protein, its function and the biological question. What is known from experiments? What do you expect to observe in the simulation, and what do you think cannot be observed on the time scale of the simulation?*

## 2. Methods

*Describe what you know about the simulation (system, length, saving interval, what was removed from the files) and the analyses you performed: which atoms you used for the superposition, which selections, which parameters (e.g. cutoffs, clustering threshold), which part of the trajectory.*

## 3. Results

### 3.1 Global properties: RMSD and radius of gyration

*Does the RMSD reach a plateau? Should the first part of the trajectory be excluded from the other analyses? Is the size of the protein stable?*

Hint: the termini and long loops can dominate the RMSD. Compare the RMSD of all the Cα atoms with that of the residues in a helix or a strand in the first frame (you can find them with DSSP, section 3.3). For Group 1, the transmembrane helices are the natural choice for the superposition.

In [ ]:
# your code here

### 3.2 Local flexibility: RMSF

*Which regions are the most flexible? What are their structural features?*

In [ ]:
# your code here

### 3.3 Secondary structure and hydrogen bonds

*Is the secondary structure stable? Which hydrogen bonds are the most stable, and which structural elements do they hold together?*

In [ ]:
# your code here

### 3.4 Conformational space: pairwise RMSD, clustering and PCA

*Does the protein visit one or more groups of conformations? How many principal components describe most of the motion, and which parts of the protein move along the first ones? Look at the free-energy landscape and check the convergence comparing the two halves of the trajectory: how much can you trust your conclusions?*

In [ ]:
# your code here

### 3.5 System-specific analysis

To answer the biological question we need to know which residues of the simulated protein form the binding sites. We find them in an experimental structure of the complex: the residues close to the ligand (or to the partner protein) form the binding site. The residue numbering of the experimental structure may differ from that of the simulation (the crystallised protein may be a fusion with another protein, or be numbered along the viral polyprotein), so we match the residues by aligning the two sequences.

Run the next cell: it defines some functions used below.

In [ ]:
import urllib.request
from Bio.Align import PairwiseAligner, substitution_matrices
from MDAnalysis.lib.util import convert_aa_code
from MDAnalysis.analysis.dssp import DSSP


def fetch_pdb(pdb_id):
    """Download a structure from the PDB (if not already here) and load it in a Universe."""
    filename = f'{pdb_id}.pdb'
    if not os.path.exists(filename):
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', filename)
    return mda.Universe(filename)


def other_molecules(structure):
    """Residue names of the molecules that are not protein or water (ligands, ions, ...)."""
    others = structure.select_atoms('not protein and not resname HOH WAT SOL TIP3')
    return sorted(set(others.resnames))


def one_letter(resname):
    try:
        return convert_aa_code(resname)
    except ValueError:
        return 'X'


def sequence(residues):
    return ''.join(one_letter(r.resname) for r in residues)


def ca_residues(atomgroup):
    """The residues of an AtomGroup that have a CA atom."""
    return atomgroup.select_atoms('protein and name CA').residues


def align_residues(residues_1, residues_2):
    """Align the sequences of two lists of residues.
    Returns the aligned pairs (residue_1, residue_2) and the fraction of identical pairs."""
    aligner = PairwiseAligner()
    aligner.mode = 'local'             # extra residues at the ends (e.g. a fusion protein) are not aligned
    aligner.substitution_matrix = substitution_matrices.load('BLOSUM62')
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(sequence(residues_1), sequence(residues_2))[0]
    pairs = []
    for (start_1, end_1), (start_2, end_2) in zip(*alignment.aligned):
        pairs += [(residues_1[start_1 + k], residues_2[start_2 + k]) for k in range(end_1 - start_1)]
    identity = np.mean([one_letter(a.resname) == one_letter(b.resname) for a, b in pairs]) if pairs else 0
    return pairs, identity


def map_to_simulation(structure, simulation):
    """Find the protein chain of an experimental structure that corresponds to the simulated protein.
    Returns the chain identifier and a dictionary {residue of the structure: residue of the simulation}."""
    sim_residues = ca_residues(simulation.atoms)
    best = None
    for chain in sorted(set(structure.select_atoms('protein').chainIDs)):
        pairs, identity = align_residues(ca_residues(structure.select_atoms(f'chainID {chain}')), sim_residues)
        print(f'chain {chain}: {len(pairs)} residues aligned to the simulation, {identity:.0%} identical')
        if best is None or len(pairs) * identity > len(best[1]) * best[2]:
            best = (chain, pairs, identity)
    chain, pairs, identity = best
    print(f'-> chain {chain} corresponds to the simulated protein')
    return chain, {a.resindex: b for a, b in pairs}


def site_residues(structure, chain, partner, cutoff=4.5):
    """Residues of the chain with a heavy atom closer than cutoff (Å) to the partner selection."""
    return structure.select_atoms(f'protein and chainID {chain} and not name H* '
                                  f'and around {cutoff} ({partner})').residues


def to_simulation(residues, mapping, simulation):
    """The residues of the simulation corresponding to residues of the experimental structure."""
    missing = [r for r in residues if r.resindex not in mapping]
    if missing:
        print('not present in the simulation:', ' '.join(f'{r.resname}{r.resid}' for r in missing))
    return simulation.residues[[mapping[r.resindex].resindex for r in residues if r.resindex in mapping]]


def to_structure(residues, mapping, structure):
    """The residues of the experimental structure corresponding to residues of the simulation
    (all of them must be present in the structure)."""
    inverse = {sim.resindex: ri for ri, sim in mapping.items()}
    return structure.residues[[inverse[r.resindex] for r in residues]]


def present_in(residues, mapping):
    """The residues of the simulation that are present in the experimental structure."""
    mapped = {sim.resindex for sim in mapping.values()}
    return residues[[r.resindex in mapped for r in residues]]


def as_selection(residues):
    """A selection string for a group of residues of the simulation."""
    return 'resindex ' + ' '.join(str(i) for i in residues.resindices)


def names(residues):
    return ' '.join(f'{r.resname}{r.resid}' for r in residues)

We also need the RMSF of each residue (Cα atoms). If you have already superimposed the trajectory in section 3.2, this does not change anything.

In [ ]:
align.AlignTraj(u, ref, select='protein and name CA', in_memory=True).run()
ca = u.select_atoms('protein and name CA')
rmsf = rms.RMSF(ca).run().results.rmsf
times = np.array([ts.time / 1000 for ts in u.trajectory])   # ns


def plot_rmsf_sites(sites):
    """RMSF of all residues with the residues of each site highlighted. sites = {'name': residues}"""
    plt.figure(figsize=(12, 4))
    plt.plot(ca.resids, rmsf, color='gray', lw=1, label='all residues')
    for site_name, residues in sites.items():
        is_site = np.isin(ca.resindices, residues.resindices)
        plt.plot(ca.resids[is_site], rmsf[is_site], 'o', label=site_name)
        print(f'{site_name}: average RMSF {rmsf[is_site].mean():.2f} Å')
    print(f'all residues: average RMSF {rmsf.mean():.2f} Å')
    plt.xlabel('Residue number')
    plt.ylabel('RMSF (Å)')
    plt.legend()
    plt.show()

## GROUP 1: AT1R

#### The ligand-binding pocket

We define the pocket as the residues within 4.5 Å of olmesartan in the inactive structure 4ZUD. ARBs share a common binding mode, so this is a reasonable definition of the pocket whatever the ligand of the simulation.

In [ ]:
inactive = fetch_pdb('4ZUD')
print('other molecules in 4ZUD:', other_molecules(inactive))
chain_inactive, map_inactive = map_to_simulation(inactive, u)

In the cell above, check that the alignment found the right chain and that most residues are identical. 4ZUD is a fusion of AT1R with another protein (BRIL), used to help crystallisation: those residues are not aligned to the simulation.

In [ ]:
pocket_xtal = site_residues(inactive, chain_inactive, 'resname OLM')
pocket = to_simulation(pocket_xtal, map_inactive, u)
print('pocket residues in 4ZUD:      ', names(pocket_xtal))
print('pocket residues in simulation:', names(pocket))

plot_rmsf_sites({'pocket': pocket})

*Which residues form the pocket? Olmesartan is anchored mainly by Tyr35, Trp84 and Arg167 (in the second extracellular loop): find them. Is the pocket more or less flexible than the rest of the receptor? Which parts of the pocket move the most?*

A simple measure of the size of the pocket is the radius of gyration of its Cα atoms. We compare it with its value in the inactive and in the active crystal structures:

In [ ]:
active = fetch_pdb('6OS0')
print('other molecules in 6OS0:', other_molecules(active))
chain_active, map_active = map_to_simulation(active, u)

# the pocket residues present in both crystal structures
common = present_in(present_in(pocket, map_inactive), map_active)
print(f'{len(common)} of the {len(pocket)} pocket residues are present in both structures')

pocket_ca = common.atoms.select_atoms('name CA')
rg_pocket = [pocket_ca.radius_of_gyration() for ts in u.trajectory]
rg_inactive = to_structure(common, map_inactive, inactive).atoms.select_atoms('name CA').radius_of_gyration()
rg_active = to_structure(common, map_active, active).atoms.select_atoms('name CA').radius_of_gyration()

plt.figure(figsize=(10, 4))
plt.plot(times, rg_pocket, label='simulation')
plt.axhline(rg_inactive, color='tab:blue', ls='--', label='inactive, 4ZUD')
plt.axhline(rg_active, color='tab:red', ls='--', label='active, 6OS0')
plt.xlabel('Time (ns)')
plt.ylabel('Radius of gyration of the pocket (Å)')
plt.legend()
plt.show()

*Does the pocket keep its size and shape? How different is the pocket in the inactive and in the active structures? Remember that the pocket of the simulated receptor contained a ligand.*

#### Is the receptor still inactive?

The hallmark of GPCR activation is an outward movement of the intracellular end of transmembrane helix 6 (TM6), which opens a cavity where the G protein binds. It is usually followed through the distance between TM3 and TM6 on the intracellular side.

To compare residues of different GPCRs, the Ballesteros-Weinstein scheme numbers each residue of a transmembrane helix as *helix.position*, where the most conserved residue of the helix is number 50. For example, the arginine of the conserved DRY motif at the end of TM3 is 3.50; in TM6, the conserved CWxP motif contains Trp 6.48, so 6.30, at the intracellular end of TM6, is 18 residues before it. We find these residues by searching the motifs in the sequence of the simulation:

In [ ]:
import re

sim_residues = ca_residues(u.atoms)
sim_sequence = sequence(sim_residues)

for motif in ['DRY', 'CW.P', 'NP..Y']:
    for match in re.finditer(motif, sim_sequence):
        print(f'{motif}: {names(sim_residues[match.start():match.end()])}')

If the motifs are found only once, the next cell defines residues 3.50 and 6.30. Otherwise, set them by hand (`r350 = sim_residues[index]`).

In [ ]:
r350 = sim_residues[re.search('DRY', sim_sequence).start() + 1]      # R of DRY
w648 = re.search('CW.P', sim_sequence).start() + 1                   # W of CWxP
r630 = sim_residues[w648 - 18]
print(f'3.50: {r350.resname}{r350.resid}   6.30: {r630.resname}{r630.resid}')

ca_350 = r350.atoms.select_atoms('name CA')[0]
ca_630 = r630.atoms.select_atoms('name CA')[0]
d_tm3_tm6 = [np.linalg.norm(ca_350.position - ca_630.position) for ts in u.trajectory]

def distance_in_structure(structure, mapping):
    """Distance between the CA atoms of the residues of the structure that correspond to 3.50 and 6.30."""
    a, b = to_structure(u.residues[[r350.resindex, r630.resindex]], mapping, structure).atoms.select_atoms('name CA')
    return np.linalg.norm(a.position - b.position)

plt.figure(figsize=(10, 4))
plt.plot(times, d_tm3_tm6, label='simulation')
plt.axhline(distance_in_structure(inactive, map_inactive), color='tab:blue', ls='--', label='inactive, 4ZUD')
plt.axhline(distance_in_structure(active, map_active), color='tab:red', ls='--', label='active, 6OS0')
plt.xlabel('Time (ns)')
plt.ylabel('Cα distance 3.50 - 6.30 (Å)')
plt.legend()
plt.show()

*Does the receptor stay inactive? How long does the activation of a GPCR take, and could you observe it in your simulation? Look at the PCA of section 3.4: does any of the first principal components involve TM6?*

**Try it:** the NPxxY motif at the end of TM7 is another switch that changes conformation upon activation. Follow the distance between the tyrosine of NPxxY (7.53) and residue 3.50 in the simulation and in the two structures.

In [ ]:
# your code here

## GROUP 2: Nsp16

#### The SAM-binding site and the Nsp10-binding surface

In the structure 6W4H we define two sites of Nsp16: the residues within 4.5 Å of SAM and those within 4.5 Å of Nsp10.

In [ ]:
complex_xtal = fetch_pdb('6W4H')
print('other molecules in 6W4H:', other_molecules(complex_xtal))
chain_16, map_16 = map_to_simulation(complex_xtal, u)

In the cell above, check that the alignment found the right chain and that most residues are identical. Note that the numbering of 6W4H follows the viral polyprotein, while that of the simulation may start from 1.

In [ ]:
sam_xtal = site_residues(complex_xtal, chain_16, 'resname SAM')
nsp10_xtal = site_residues(complex_xtal, chain_16, f'protein and not chainID {chain_16}')

sam_site = to_simulation(sam_xtal, map_16, u)
nsp10_site = to_simulation(nsp10_xtal, map_16, u)
print('SAM site:            ', names(sam_site))
print('Nsp10-binding surface:', names(nsp10_site))
print('in common:            ', names(sam_site.intersection(nsp10_site)))

plot_rmsf_sites({'SAM site': sam_site, 'Nsp10-binding surface': nsp10_site})

*Are the two sites more or less flexible than the rest of the protein? Which parts of the sites move the most? Are they far from each other in the structure?*

Let's follow how much each site deviates from the starting structure: we superimpose the whole protein and compute the RMSD of the Cα atoms of each site.

In [ ]:
rmsd_sites = rms.RMSD(u, ref, select='protein and name CA',
                      groupselections=[f'name CA and ({as_selection(sam_site)})',
                                       f'name CA and ({as_selection(nsp10_site)})']).run().results.rmsd

plt.figure(figsize=(10, 4))
plt.plot(times, rmsd_sites[:, 2], label='whole protein')
plt.plot(times, rmsd_sites[:, 3], label='SAM site')
plt.plot(times, rmsd_sites[:, 4], label='Nsp10-binding surface')
plt.xlabel('Time (ns)')
plt.ylabel('Cα RMSD (Å)')
plt.legend()
plt.show()

*Do the sites keep the conformation they have in the complex? Experimentally, Nsp10 is needed for the activity of Nsp16 and increases its affinity for SAM: is this consistent with what you see? What simulation would you run to test your hypothesis?*

**Try it:**

1. The catalytic site of Nsp16 is formed by the tetrad Lys46, Asp130, Lys170, Glu203 (numbering of Nsp16). Find these residues in your simulation (check the numbering!) and follow the distances between their side chains.
2. The structure 6WVN also contains the cap analogue. Use `other_molecules` to find its residue name, define the cap-binding site in the same way and analyse its flexibility.

In [ ]:
# your code here

### 3.6 Visualisation

*Look at the trajectory and connect what you see with the analyses above. Use pictures in the report where they help.*

The next cell shows 50 frames of the trajectory. The residues in `highlight` are shown as sticks: set it to one of the sites of section 3.5 (e.g. `pocket`, `sam_site`, `nsp10_site`).

In [ ]:
highlight = None      # e.g. pocket, sam_site, nsp10_site

step = max(1, u.trajectory.n_frames // 50)
protein.write('movie.pdb', frames=u.trajectory[::step])

view = py3Dmol.view(width=700, height=500)
view.addModelsAsFrames(open('movie.pdb').read(), 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
if highlight is not None:
    view.addStyle({'resi': [int(r) for r in highlight.resids]}, {'stick': {}})
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 100})
view.show()

## 4. Discussion

*Interpret the results and answer the question of your group. Some points to consider:*

*   *Which of your observations are robust, and which could change with a longer simulation or with independent replicas? Use the convergence analysis of section 3.4.*
*   *What can, and what cannot, be learned from a simulation of this length about the function of the protein?*
*   *Which additional simulations would you run to answer the question better?*

## 5. Conclusions

## References

*   Michaud-Agrawal N, Denning EJ, Woolf TB, Beckstein O (2011) MDAnalysis: a toolkit for the analysis of molecular dynamics simulations. *J Comput Chem* 32:2319-2327
*   Ballesteros JA, Weinstein H (1995) Integrated methods for the construction of three-dimensional models and computational probing of structure-function relations in G protein-coupled receptors. *Methods Neurosci* 25:366-428
*   Zhang H et al. (2015) Structure of the angiotensin receptor revealed by serial femtosecond crystallography. *Cell* 161:833-844 (4YAY)
*   Zhang H et al. (2015) *J Biol Chem* (4ZUD)
*   Wingler LM et al. (2020) Angiotensin and biased analogs induce structurally distinct active conformations within a GPCR. *Science* 367:888-892 (6OS0)
*   Rosas-Lemus M et al. (2020) High-resolution structures of the SARS-CoV-2 2'-O-methyltransferase reveal strategies for structure-based inhibitor design. *Sci Signal* 13:eabe1202 (6W4H, 6WVN)